### 09. Building a Hybrid Recommendation Model

**Objective:** Combine **Content-Based Filtering** (Task 6) and **Collaborative Filtering** (Task 8) into a single hybrid recommender that leverages both TED Talk content and user preferences.

**Inputs:**
- `../data/processed/ted_talks_cleaned.csv` (cleaned TED dataset)
- CB artefacts (Task 6): `tfidf_matrix_exp3.npz`, `cosine_sim_exp3.npy`, `tfidf_vectorizer_exp3.joblib`
- CF model (Task 8): `cf_model.joblib` (`user_item`, `user_means`, `sim_df`, `k`)
- `../data/interaction/user_interactions.csv` (for CF rebuild fallback)

**Data:** 500 users x 200 talks, ~9,800 ratings (~90% sparse). At this density CF produces recommendations for every user, so the hybrid has two live signals to blend.

**Approach:**
1. Load CB artefacts and the mean-centered CF model; rebuild once if missing.
2. For a given `user_id` + `seed_title`:
   - CB recommends talks similar to `seed_title`.
   - CF predicts mean-centered ratings for `user_id`'s neighbours and clips to [1, 5].
3. **Exclude talks the user has already rated** from both lists *before* normalizing, so the
   `[0, 1]` range reflects only recommendable candidates.
4. **Min-max normalize** each model's scores to `[0, 1]`.
5. **CF confidence:** `cf_conf = min(n_raters / 5, 1)` (0 for talks outside the CF universe).
6. **Combine:** `hybrid = α × CB_norm + (1 − α) × CF_norm × CF_conf` (default **α = 0.5**).
7. **Merge duplicates** (same `talk_id` in both lists -> combine normalized contributions).
8. Rank and return top-N.

In [51]:
import numpy as np
import pandas as pd
import os
import difflib
import joblib
from pathlib import Path
from scipy.sparse import load_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

PROCESSED_DIR = Path("../data/processed")
INTERACTION_DIR = Path("../data/interaction")

##### Load Cleaned Talks Dataset

In [52]:
df_talks = pd.read_csv(PROCESSED_DIR / "ted_talks_cleaned.csv")
print(f"Talks: {df_talks.shape[0]} rows × {df_talks.shape[1]} columns")
df_talks[["talk_id", "title", "tags"]].head(2)

Talks: 4005 rows × 16 columns


,talk_id,title,tags
0,1,averting the climate crisis,"alternative energy, cars, climate change, culture, environment, global issues, science, sustaina..."
1,92,the best stats you ve ever seen,"africa, asia, google, demo, economics, global issues, health, statistics, global development, vi..."


##### Load Content-Based Model (Task 6 - Exp3)

Loads the saved TF-IDF matrix, cosine similarity matrix, and vectorizer. If any are missing, rebuilds them once from `combined_text_prefixed` using the same config as Task 6 Exp3 (**Title + Description + Tags**).

In [53]:
CB_TFIDF_PATH   = PROCESSED_DIR / "tfidf_matrix_exp3.npz"
CB_COSIM_PATH   = PROCESSED_DIR / "cosine_sim_exp3.npy"
CB_VEC_PATH     = PROCESSED_DIR / "tfidf_vectorizer_exp3.joblib"

if CB_TFIDF_PATH.exists() and CB_COSIM_PATH.exists() and CB_VEC_PATH.exists():
    print("Loading CB artefacts (Task 6)...")
    cb_tfidf = load_npz(CB_TFIDF_PATH)
    cb_cosim = np.load(CB_COSIM_PATH)
    cb_vectorizer = joblib.load(CB_VEC_PATH)
else:
    print("CB artefacts missing - rebuilding (Exp3: Title + Description + Tags)...")
    cb_vectorizer = TfidfVectorizer(
        stop_words="english", ngram_range=(1, 2), min_df=2, max_df=0.9,
    )
    combined = (
        df_talks["title"].fillna("") + " " +
        df_talks["description"].fillna("") + " " +
        df_talks["tags"].fillna("").str.replace(",", " ", regex=False)
    )
    cb_tfidf = cb_vectorizer.fit_transform(combined)
    cb_cosim = cosine_similarity(cb_tfidf, dense_output=True)
    np.fill_diagonal(cb_cosim, 0.0)

    from scipy.sparse import save_npz
    save_npz(CB_TFIDF_PATH, cb_tfidf)
    np.save(CB_COSIM_PATH, cb_cosim)
    joblib.dump(cb_vectorizer, CB_VEC_PATH)
    print("CB artefacts rebuilt and saved.")

# Align df_talks row order with cb_cosim indices (must be identical)
cb_df = df_talks.reset_index(drop=True)
print(f"CB TF-IDF shape:        {cb_tfidf.shape}")
print(f"CB cosine matrix shape: {cb_cosim.shape}")

Loading CB artefacts (Task 6)...
CB TF-IDF shape:        (4005, 23643)
CB cosine matrix shape: (4005, 4005)


##### Load Collaborative Filtering Model (Task 8)

Loads the `cf_model.joblib` bundle: the user-item matrix, per-user means, the mean-centered
similarity matrix, and the neighbour count `k`. If the bundle is missing, it is rebuilt from
`user_interactions.csv` using the same mean-centering logic as Task 8.

In [54]:
CF_MODEL_PATH = PROCESSED_DIR / "cf_model.joblib"

if CF_MODEL_PATH.exists():
    print("Loading CF model (Task 8)...")
    cf_model = joblib.load(CF_MODEL_PATH)
    user_item_matrix = cf_model["user_item"]
    user_means = cf_model["user_means"]
    user_sim_df = cf_model["sim_df"]
    K_NEIGHBORS = int(cf_model.get("k", 20))
else:
    print("CF model missing - rebuilding mean-centered model from user_interactions.csv...")
    interactions = pd.read_csv(INTERACTION_DIR / "user_interactions.csv")
    user_item_matrix = interactions.pivot_table(
        index="user_id", columns="talk_id", values="rating", aggfunc="mean",
    )
    user_means = user_item_matrix.mean(axis=1)
    centered = user_item_matrix.sub(user_means, axis=0).fillna(0.0)
    sim = cosine_similarity(centered.values, dense_output=True)
    np.fill_diagonal(sim, 0.0)
    user_sim_df = pd.DataFrame(sim, index=user_item_matrix.index, columns=user_item_matrix.index)
    K_NEIGHBORS = 20
    cf_model = {"user_item": user_item_matrix, "user_means": user_means, "sim_df": user_sim_df, "k": K_NEIGHBORS}
    joblib.dump(cf_model, CF_MODEL_PATH)
    print("CF model rebuilt and saved.")

# Mean-centered matrix used by the recommender
centered_matrix = user_item_matrix.sub(user_means, axis=0).fillna(0.0)

print(f"User-item matrix shape: {user_item_matrix.shape}")
print(f"User similarity shape:  {user_sim_df.shape}")
print(f"Neighbours (k):         {K_NEIGHBORS}")

Loading CF model (Task 8)...
User-item matrix shape: (500, 200)
User similarity shape:  (500, 500)
Neighbours (k):         20


##### Content-Based Recommender (per seed title)

Same matching logic as Task 5/6: exact -> partial -> invalid (with fuzzy suggestion). Returns `talk_id`, `title`, raw CB cosine score.

In [55]:
def cb_recommend(seed_title, top_n=10):
    """
    Content-based recommendations for a seed title (Exp3 pipeline).
    Returns DataFrame: [talk_id, title, cb_score] or None if invalid.
    """
    title_clean = seed_title.strip().lower()
    titles_lower = cb_df["title"].str.lower()

    matches = cb_df.index[titles_lower == title_clean].tolist()

    if not matches:
        partial = cb_df.index[titles_lower.str.contains(title_clean, regex=False, na=False)].tolist()
        if len(partial) == 1:
            matches = partial
        elif len(partial) > 1:
            best = difflib.get_close_matches(seed_title, cb_df.loc[partial, "title"].tolist(), n=1, cutoff=0.0)
            if best:
                matches = [cb_df.index[cb_df["title"] == best[0]].tolist()[0]]

    if not matches:
        sug = difflib.get_close_matches(seed_title, cb_df["title"].tolist(), n=1, cutoff=0.5)
        if sug:
            print(f"Invalid seed title: '{seed_title}' - did you mean '{sug[0]}'?")
        else:
            print(f"Invalid seed title: '{seed_title}' - no suggestions.")
        return None

    seed_idx = matches[0]
    scores = cb_cosim[seed_idx]
    top_idx = [i for i in scores.argsort()[::-1] if i != seed_idx][:top_n]

    return pd.DataFrame({
        "talk_id":  [int(cb_df.loc[i, "talk_id"]) for i in top_idx],
        "title":    [cb_df.loc[i, "title"] for i in top_idx],
        "cb_score": [round(float(scores[i]), 4) for i in top_idx],
    })

##### Collaborative Filtering Recommender (per user)

Mean-centered user-based CF, same logic as Task 8's `recommend_for_user()`: top-K neighbours,
similarity-weighted deviations from neighbours' means, added back to the target user's mean and
clipped to **[1, 5]**. Returns `talk_id`, `title`, `cf_score`, and `n_raters` (number of neighbours
who rated the talk), sorted by `cf_score` then `n_raters` (both descending).

In [56]:
def cf_recommend(user_id, top_n=10, k_neighbors=None):
    """
    Mean-centered user-based CF recommender.

    Returns DataFrame: [talk_id, title, cf_score, n_raters] or None if invalid/cold-start.
    Predictions are clipped to [1, 5] and sorted by cf_score then n_raters (desc).
    """
    if k_neighbors is None:
        k_neighbors = K_NEIGHBORS
    if user_id not in user_item_matrix.index:
        print(f"Invalid user_id: '{user_id}'.")
        return None

    user_ratings = user_item_matrix.loc[user_id]
    if user_ratings.dropna().empty:
        print(f"User '{user_id}' has no interaction history - cold-start.")
        return None

    sims = user_sim_df.loc[user_id].drop(user_id)
    neighbours = sims.sort_values(ascending=False).head(k_neighbors)
    neighbour_ids = neighbours.index
    neighbour_weights = neighbours.values

    neighbour_centered = centered_matrix.loc[neighbour_ids]
    candidate_mask = user_ratings.isna()
    neighbours_with_data = user_item_matrix.loc[neighbour_ids].notna()
    target_mean = float(user_means[user_id])

    pred_data = []
    for talk_id in user_item_matrix.columns:
        if not candidate_mask[talk_id]:
            continue
        rated_by = neighbours_with_data[talk_id]
        if not rated_by.any():
            continue
        weights = neighbour_weights[rated_by.values]
        centered_ratings = neighbour_centered.loc[rated_by, talk_id].values
        denom = np.sum(np.abs(weights))
        if denom <= 0:
            continue
        pred_dev = np.dot(weights, centered_ratings) / denom
        pred_rating = float(np.clip(target_mean + pred_dev, 1.0, 5.0))
        pred_data.append((talk_id, pred_rating, int(rated_by.sum())))

    if not pred_data:
        print(f"No CF recommendations for '{user_id}'.")
        return None

    pred_df = pd.DataFrame(pred_data, columns=["talk_id", "cf_score", "n_raters"])
    pred_df = pred_df.sort_values(["cf_score", "n_raters"], ascending=[False, False]).head(top_n)

    pred_df = pred_df.merge(df_talks[["talk_id", "title"]], on="talk_id", how="left")
    pred_df["cf_score"] = pred_df["cf_score"].round(4)
    return pred_df[["talk_id", "title", "cf_score", "n_raters"]].reset_index(drop=True)

##### Hybrid Recommender

**Steps:**
1. Get CB top-N and CF top-N for the same `user_id` + `seed_title`.
2. **Exclude talks the user has already rated** from both lists *before* normalizing.
3. Min-max normalize each remaining list's scores to `[0, 1]`.
4. Add **CF confidence**: `cf_conf = min(n_raters / 5, 1)`; talks outside the CF universe get 0.
5. Concatenate both lists and **merge duplicates** (same `talk_id` -> combine contributions).
6. `hybrid_score = α × cb_norm + (1 − α) × cf_norm × cf_conf`.
   - A talk missing from one side contributes 0 for that side.
7. Sort by `hybrid_score`, return top-N.

**Default α = 0.5** (equal weight). Adjustable.

> **Why filter before normalizing?** Removing already-rated talks *after* min-max shifts the
> min/max endpoints, so the surviving scores no longer span `[0, 1]` and α stops meaning what it
> should. Filtering first keeps the normalization range honest.

> **Why confidence-weight CF?** A prediction backed by 1 neighbour is far less reliable than one
> backed by 5+. Scaling CF by `min(n_raters/5, 1)` down-weights thin-evidence predictions and
> zeroes out talks that CF never scored.

In [57]:
def _min_max(s):
    """Min-max normalize a pandas Series to [0, 1]. Handles empty/constant series."""
    if len(s) == 0:
        return s
    lo, hi = s.min(), s.max()
    if hi - lo < 1e-12:
        return pd.Series(np.ones(len(s)), index=s.index)
    return (s - lo) / (hi - lo)


def hybrid_recommend(user_id, seed_title, top_n=5, alpha=0.5, candidate_pool=20):
    """
    Hybrid recommender combining content-based and mean-centered collaborative filtering.

    Fusion: hybrid = alpha * cb_norm + (1 - alpha) * cf_norm * cf_conf
    where cf_conf = min(n_raters / 5, 1.0) (0 for talks outside the CF universe).

    Parameters:
    - user_id (str): Target user.
    - seed_title (str): Seed TED Talk title (used by content-based model).
    - top_n (int): Number of final recommendations.
    - alpha (float): Weight on CB score in [0, 1]. 1 = CB only, 0 = CF only.
    - candidate_pool (int): How many top candidates to pull from each model before merging.

    Returns:
    - pd.DataFrame: [Rank, talk_id, title, cb_score, cf_score, hybrid_score]
    - None: if either model fails.
    """
    if not (0 <= alpha <= 1):
        print("Error: alpha must be between 0 and 1.")
        return None
    if user_id not in user_item_matrix.index:
        print(f"Invalid user_id: '{user_id}'.")
        return None

    # 1. Get candidates
    cb_recs = cb_recommend(seed_title, top_n=candidate_pool)
    cf_recs = cf_recommend(user_id, top_n=candidate_pool)
    if cb_recs is None or cf_recs is None:
        return None

    # 2. Exclude already-rated talks BEFORE normalization so the [0, 1]
    #    range is computed only over candidates that can be recommended.
    rated = set(user_item_matrix.loc[user_id].dropna().index)
    cb_recs = cb_recs[~cb_recs["talk_id"].isin(rated)].copy()
    cf_recs = cf_recs[~cf_recs["talk_id"].isin(rated)].copy()

    if cb_recs.empty and cf_recs.empty:
        print(f"No candidate talks left for '{user_id}'.")
        return None

    # 3. Normalize (after filtering) + CF confidence
    cb_recs["cb_norm"] = _min_max(cb_recs["cb_score"])
    cf_recs["cf_norm"] = _min_max(cf_recs["cf_score"])
    cf_recs["cf_conf"] = np.minimum(cf_recs["n_raters"] / 5.0, 1.0)

    # 4. Merge on talk_id (outer join keeps talks from either model)
    merged = pd.merge(
        cb_recs[["talk_id", "title", "cb_score", "cb_norm"]],
        cf_recs[["talk_id", "title", "cf_score", "cf_norm", "cf_conf"]],
        on="talk_id", how="outer", suffixes=("", "_cf"),
    )

    # Coalesce title (prefer CB title; fallback to CF)
    merged["title"] = merged["title"].fillna(merged["title_cf"])
    merged = merged.drop(columns=["title_cf"])

    # Talks appearing in only one list -> missing side = 0.
    # Talks outside the CF universe get cf_conf = 0, so they receive no CF contribution.
    merged["cb_norm"] = merged["cb_norm"].fillna(0.0)
    merged["cf_norm"] = merged["cf_norm"].fillna(0.0)
    merged["cf_conf"] = merged["cf_conf"].fillna(0.0)

    # 5. Confidence-weighted hybrid score
    merged["hybrid_score"] = alpha * merged["cb_norm"] + (1 - alpha) * merged["cf_norm"] * merged["cf_conf"]

    # Fill missing raw scores with NaN for transparency
    merged["cb_score"] = merged["cb_score"].fillna(np.nan)
    merged["cf_score"] = merged["cf_score"].fillna(np.nan)

    # 6. Rank and return top-N
    merged = merged.sort_values("hybrid_score", ascending=False).head(top_n).reset_index(drop=True)
    merged.insert(0, "Rank", range(1, len(merged) + 1))
    merged["hybrid_score"] = merged["hybrid_score"].round(4)

    return merged[["Rank", "talk_id", "title", "cb_score", "cf_score", "hybrid_score"]]

##### Compare Individual Models vs. Hybrid

Pick a `user_id` and a `seed_title` that this user has interacted with (so both models have meaningful signal). We test 3 combinations.

In [58]:
# Helper: show the user's top-rated talks (useful for choosing a seed)
def user_top_rated(user_id, n=5):
    row = user_item_matrix.loc[user_id].dropna().sort_values(ascending=False).head(n)
    return pd.DataFrame({
        "talk_id": row.index,
        "rating": row.values,
    }).merge(df_talks[["talk_id", "title"]], on="talk_id", how="left")

user_top_rated("U0001")

,talk_id,rating,title
0,13029,4.0,the dead zone of the gulf of mexico
1,1317,3.0,can astronomers help doctors
2,771,3.0,four ways to fix a broken legal system
3,2253,3.0,why city flags may be the worst designed thing you ve never noticed
4,27155,3.0,how to motivate people to do good for others


In [59]:
# Comparison runner
def compare_models(user_id, seed_title, top_n=5, alpha=0.5):
    print(f"\n{'='*100}")
    print(f"USER: {user_id} | SEED TALK: '{seed_title}' | α = {alpha}")
    print(f"{'='*100}")

    print("\n--- Content-Based (CB) ---")
    cb = cb_recommend(seed_title, top_n=top_n)
    print(cb.to_string(index=False) if cb is not None else "None")

    print("\n--- Collaborative Filtering (CF) ---")
    cf = cf_recommend(user_id, top_n=top_n)
    print(cf.to_string(index=False) if cf is not None else "None")

    print("\n--- Hybrid (CB + CF) ---")
    hy = hybrid_recommend(user_id, seed_title, top_n=top_n, alpha=alpha)
    print(hy.to_string(index=False) if hy is not None else "None")

In [60]:
# Test 1
seed = user_top_rated("U0001", 1).iloc[0]["title"]
print(f"Seed (U0001's highest-rated talk): {seed!r}")
compare_models("U0001", seed, top_n=5, alpha=0.5)

Seed (U0001's highest-rated talk): 'the dead zone of the gulf of mexico'

USER: U0001 | SEED TALK: 'the dead zone of the gulf of mexico' | α = 0.5

--- Content-Based (CB) ---
 talk_id                                                    title  cb_score
   14485 the discoveries awaiting us in the ocean s twilight zone    0.1408
    1054                                         addicted to risk    0.1285
    1499                  a novel idea for cleaning up oil spills    0.1251
     940                           the political chemistry of oil    0.1249
     471                     a dive into the reef s twilight zone    0.1176

--- Collaborative Filtering (CF) ---
 talk_id                                                title  cf_score  n_raters
     912                  global population growth box by box    3.6303         1
   10975 you don t have to be an expert to solve big problems    3.3198         4
    1534             a test for parkinson s with a phone call    3.3083         1
   

In [61]:
# Test 2
seed = user_top_rated("U0002", 1).iloc[0]["title"]
print(f"Seed (U0002's highest-rated talk): {seed!r}")
compare_models("U0002", seed, top_n=5, alpha=0.5)

Seed (U0002's highest-rated talk): 'the year i was homeless'

USER: U0002 | SEED TALK: 'the year i was homeless' | α = 0.5

--- Content-Based (CB) ---
 talk_id                                                     title  cb_score
   13022           how the arts help homeless youth heal and build    0.2710
    3620                the housing first approach to homelessness    0.2621
    2840 a practical way to help the homeless find work and safety    0.2137
    1894                            depression the secret we share    0.1667
    1797               how a penny made me feel like a millionaire    0.1293

--- Collaborative Filtering (CF) ---
 talk_id                                       title  cf_score  n_raters
      91            invest in africa s own solutions    4.7186         1
   32250   will there ever be a mile high skyscraper    4.2102         1
    1773            the dangers of willful blindness    4.0365         3
    2707 a burial practice that nourishes the planet    4

In [62]:
# Test 3
seed = user_top_rated("U0003", 1).iloc[0]["title"]
print(f"Seed (U0003's highest-rated talk): {seed!r}")
compare_models("U0003", seed, top_n=5, alpha=0.5)

Seed (U0003's highest-rated talk): 'comics that ask what if'

USER: U0003 | SEED TALK: 'comics that ask what if' | α = 0.5

--- Content-Based (CB) ---
 talk_id                                    title  cb_score
    2186                          math is forever    0.1514
    1276                    science versus wonder    0.1472
    1386    questions no one knows the answers to    0.1438
    1963 the discovery that could rewrite physics    0.1403
     118                    the genesis of google    0.1384

--- Collaborative Filtering (CF) ---
 talk_id                                        title  cf_score  n_raters
   42461                                 new colossus    4.6234         1
   55793           could a breathalyzer detect cancer    3.8643         1
    1656                     a new kind of job market    3.8262         2
    2508                 the beauty of being a misfit    3.7964         2
   27155 how to motivate people to do good for others    3.7249         2

--- Hy

##### Alpha Sensitivity Test

How does changing α shift the hybrid output? α = 1.0 -> CB only; α = 0.0 -> CF only
(confidence-weighted, so its order can differ from the raw `cf_recommend` ranking).

In [63]:
seed_u1 = user_top_rated("U0001", 1).iloc[0]["title"]

for a in [0.0, 0.3, 0.5, 0.7, 1.0]:
    print(f"\n{'='*100}")
    print(f"α = {a}  ({'CF only' if a == 0 else 'CB only' if a == 1 else 'balanced'})")
    print(f"{'='*100}")
    hy = hybrid_recommend("U0001", seed_u1, top_n=5, alpha=a)
    if hy is None:
        print("None")
    else:
        print(hy[["Rank", "talk_id", "title", "hybrid_score"]].to_string(index=False))


α = 0.0  (CF only)
 Rank  talk_id                                                       title  hybrid_score
    1    10975        you don t have to be an expert to solve big problems        0.4305
    2      613                          how to make filthy water drinkable        0.2119
    3      912                         global population growth box by box        0.2000
    4      270                                          the bottom billion        0.1558
    5    56012 climate change will displace millions here s how we prepare        0.1336

α = 0.3  (balanced)
 Rank  talk_id                                                    title  hybrid_score
    1    10975     you don t have to be an expert to solve big problems        0.3014
    2    14485 the discoveries awaiting us in the ocean s twilight zone        0.3000
    3     1054                                         addicted to risk        0.2357
    4     1499                  a novel idea for cleaning up oil spills        0.

##### Invalid Input Tests

In [64]:
# Invalid user
hybrid_recommend("U9999", "the story of ezra", top_n=5)

Invalid user_id: 'U9999'.


In [65]:
# Invalid seed title
hybrid_recommend("U0001", "this talk does not exist anywhere", top_n=5)

Invalid seed title: 'this talk does not exist anywhere' - did you mean 'the walk from no to yes'?


In [66]:
# Invalid alpha
hybrid_recommend("U0001", seed_u1, top_n=5, alpha=1.5)

Error: alpha must be between 0 and 1.


##### Observations

**How the hybrid combines the models:**

1. **Coverage.** CB recommendations stay close to the seed talk; CF recommendations reflect what similar users rated. The outer merge keeps both candidate sets, so a talk surfaced by either model can reach the final list.

2. **Confidence-weighted CF.** A CF prediction backed by one neighbour is much less reliable than one backed by five. `cf_conf = min(n_raters / 5, 1)` scales each CF contribution by how much neighbour evidence supports it, and talks outside the CF universe get 0. At 90% sparsity most CF candidates have only 1–4 raters, so the CF branch is deliberately discounted.

3. **Normalization order fixed.** Already-rated talks are now removed *before* min-max. Previously they were removed after, which shifted the min/max endpoints and meant the surviving scores no longer spanned `[0, 1]`. Filtering first keeps α interpretable.

4. **Duplicate merging.** When a talk appears in both CB and CF lists, both normalized contributions are combined - "both models agree" produces a stronger signal than either alone.

5. **Exclusion of already-rated talks.** No talk the user has rated is ever recommended.

**What the α sweep shows (U0001):**

- **α = 1.0** returns the pure CB list; **α = 0.0** returns CF only, confidence-weighted (`cf_norm × cf_conf`), so its order differs from the raw `cf_recommend` ranking.
- At **α = 0.5**, U0001's top-5 are all CB. The strongest raw CF candidate (912) is backed by only **1** neighbour, so `cf_conf = 0.2` and its contribution (`0.5 × 1.0 × 0.2 = 0.10`) falls below the CB scores (~0.30–0.50).
- At **α = 0.3**, CF talk **10975** (n_raters = 4 -> `cf_conf = 0.8`) narrowly leads (`0.3014` vs `0.3000`), showing that a well-supported CF prediction can still compete.
- U0002's rank-5 hybrid item (**1773**) is CF-backed by **3** neighbours and survives at α = 0.5.

**Limitations observed:**

1. **Seed dependency.** The hybrid still requires a `seed_title` for the CB side. If the seed doesn't exist, CB fails and the function returns `None` even though CF alone would still be useful.
   - *Mitigation in Task 10:* build a multi-talk CB profile from the user's own high ratings instead of a single seed.

2. **Cold-start:** a user with no interaction history can't get a hybrid rec (CF fails). CB alone could serve them via a seed title - a potential fallback.

3. **Confidence weighting is coarse.** `n_raters / 5` saturates at five raters, and at 90% sparsity most candidates have 1–4 raters, so CF is heavily discounted. A smoother confidence estimate (e.g., shrink toward the user mean) could be better.

4. **Min-max resolution on a small pool.** With `candidate_pool=20`, min-max has limited resolution and can over-weight small differences. Mean-centering made CF scores graded rather than tied (U0001's top-5 span **3.30–3.63**), but a larger pool would smooth rankings further.

5. **α = 0.5 is not always optimal.** For users with dense rating histories, CF may deserve higher weight (α < 0.5). For sparse users, CB should dominate (α > 0.5). Task 10 tunes α on held-out data.

**Next step (Task 10):** Evaluate CB, CF, and the hybrid on a held-out set and choose α by measured Precision@K / Recall@K / NDCG@K.

> **Note on scale:** the α behavior shown here is per-user. Task 10's held-out 
> evaluation across 489 users finds the hybrid's best α is 1.0 (i.e., pure CB) - 
> at the dataset level, the CF branch does not add measurable value on this 
> data. See Task 10 for the full comparison.

#### 📄 Task Report

The written report for this notebook lives at [`reports/task_09_observations.md`](../reports/task_09_observations.md).

- Markdown: [`reports/task_09_observations.md`](../reports/task_09_observations.md)
- PDF: [`reports/pdf/task_09_observations.pdf`](../reports/pdf/task_09_observations.pdf)

If a link does not open inside your notebook viewer, open the file directly from the project root.
